# Create Base Order Dataset

Your first responsibility is to create the first clean, order-level dataset. The final ML model should have one row per order. Be careful when joining order items because one order can have multiple products. First, aggregate item-level data, then join it with the order table.

Output: `data/processed/base_order_dataset.csv`

## Join Logic

- Load the minimum Olist tables: orders, customers, order_items, sellers, products.
- Clean dates, numeric columns, zip prefixes, and string fields.
- Join `order_items` to `products` and `sellers` while still at item level.
- Aggregate the item-level table to `order_id` before joining with orders.
- Join orders + customers + aggregated item/product/seller features.
- Validate that the final dataset has one row per order.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_config, load_olist_tables
from src.data.clean_data import (
    clean_olist_tables,
    aggregate_order_items_to_order_level,
    build_base_order_dataset,
    save_base_order_dataset,
)

In [ ]:
config = load_config(PROJECT_ROOT / "config.yaml")

raw_data = load_olist_tables(
    raw_dir=PROJECT_ROOT / config["paths"]["raw_dir"],
    table_files=config["files"],
    required_tables=["orders", "customers", "order_items", "sellers", "products"],
)

{table: df.shape for table, df in raw_data.items()}

In [ ]:
clean_data = clean_olist_tables(raw_data)
{table: df.shape for table, df in clean_data.items()}

In [ ]:
item_features = aggregate_order_items_to_order_level(
    order_items=clean_data["order_items"],
    products=clean_data["products"],
    sellers=clean_data["sellers"],
)

item_features.head()

In [ ]:
base_orders = build_base_order_dataset(clean_data)

print(f"Rows: {base_orders.shape[0]:,}")
print(f"Columns: {base_orders.shape[1]:,}")
print(f"Duplicate order_id rows: {base_orders['order_id'].duplicated().sum():,}")

base_orders.head()

In [ ]:
assert base_orders["order_id"].is_unique, "Base dataset must have one row per order."
assert len(base_orders) == clean_data["orders"]["order_id"].nunique(), "Base row count should match unique orders."

output_path = PROJECT_ROOT / config["paths"]["processed_dir"] / config["outputs"]["base_order_dataset"]
save_base_order_dataset(base_orders, output_path)

output_path